# 06 - So sanh Face Recognition Models (ArcFace vs FaceNet vs MobileFaceNet vs antelopev2 vs AdaFace vs SFace)

**Muc tieu:** notebook nay so sanh **6 model trich xuat embedding khuon mat** (khong phai detection) de
co bo benchmark ro rang hon cho backend nhan dang.

**6 model duoc so sanh:**

| Model | Backbone / model pack | Loss | Embedding | Vai tro trong so sanh |
|---|---|---|---:|---|
| **ArcFace** (`insightface`, `buffalo_l`) | ResNet50@WebFace600K | ArcFace | 512 | **Baseline chinh**, dang dung trong backend |
| **FaceNet** (`facenet-pytorch`) | InceptionResnetV1 | Triplet Loss | 512 | Dai dien mot huong metric learning kinh dien, khac han ArcFace |
| **MobileFaceNet** (`insightface`, `buffalo_sc`) | MBF@WebFace600K | ArcFace | 512 | Mo hinh nhe, phu hop edge/mobile |
| **antelopev2** (`insightface`) | ResNet100@Glint360K | ArcFace | 512 | Mo hinh manh, do chinh xac cao |
| **AdaFace** (`uniface`, IR_101) | IR-101 | AdaFace | 512 | Mo hinh thich nghi voi chat luong anh |
| **SFace** (`OpenCV FaceRecognizerSF`) | OpenCV DNN / SFace ONNX | SFace Loss | 128 | Dai dien OpenCV, dung de so sanh voi ecosystem khac |

**Ghi chu:**  
- ArcFace / MobileFaceNet / antelopev2 dung chung `insightface` va `onnxruntime-gpu` neu co CUDAExecutionProvider.  
- FaceNet dung `facenet-pytorch`.  
- AdaFace dung `uniface` (se auto download weights).  
- SFace dung model ONNX cua OpenCV Zoo va `cv2.FaceRecognizerSF`.

## Phan A - Cai dat thu vien va tai model

- `insightface` da co tu Notebook 01/02, dung cho ArcFace / MobileFaceNet / antelopev2.
- `facenet-pytorch` dung cho FaceNet.
- `uniface` dung cho AdaFace.
- `opencv-python>=4.10` dung cho `cv2.FaceRecognizerSF` (SFace).

**CANH BAO QUAN TRONG ve `facenet-pytorch`:** package nay co metadata pin torch cu hon.  
De tranh pip keo torch xuong ban khong phu hop voi RTX 5070, cai `facenet-pytorch` bang `--no-deps`.

In [1]:
# ===== CELL A1: Cai dat FaceNet + UniFace =====
# facenet-pytorch pin torch<2.3 trong metadata -> dung --no-deps de KHONG bi pip ha cap torch.
get_ipython().system("pip install -q --no-deps facenet-pytorch")
get_ipython().system("pip install -q uniface requests tqdm")

print("Da cai: facenet-pytorch --no-deps (FaceNet)")
print("Da cai: uniface (AdaFace)")
print("insightface da co tu Notebook 01/02, dung chung cho ArcFace / MobileFaceNet / antelopev2")


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Da cai: facenet-pytorch --no-deps (FaceNet)
Da cai: uniface (AdaFace)
insightface da co tu Notebook 01/02, dung chung cho ArcFace / MobileFaceNet / antelopev2



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# ===== CELL A1b: Xac nhan torch KHONG bi ha cap sau khi cai facenet-pytorch =====
import torch
print(f"Torch version sau khi cai facenet-pytorch: {torch.__version__}")
if torch.__version__.startswith(("2.0.", "2.1.", "2.2.")):
    print("CANH BAO: torch da bi ha cap xuong ban cu (< 2.3) - can go va cai lai "
          "torch>=2.7 (cu128) de chay duoc CUDA tren RTX 5070.")
else:
    print("Torch van dang o ban moi, chua bi facenet-pytorch keo xuong. On.")

Torch version sau khi cai facenet-pytorch: 2.11.0+cu128
Torch van dang o ban moi, chua bi facenet-pytorch keo xuong. On.


In [3]:
# ===== CELL A2: Duong dan project (giong quy uoc cac notebook truoc, dung lai gallery/probe) =====
import os
import pandas as pd

BASE_DIR = r'D:\face-security-system'
AI_DIR = os.path.join(BASE_DIR, 'ai')

SPLIT_GALLERY = os.path.join(AI_DIR, 'dataset', 'splits', 'gallery')   # anh align 112x112, tu Notebook 01
SPLIT_PROBE   = os.path.join(AI_DIR, 'dataset', 'splits', 'probe')     # anh align 112x112, tu Notebook 01
MODELS_DIR    = os.path.join(AI_DIR, 'models')
RESULTS_DIR   = os.path.join(AI_DIR, 'dataset', 'evaluation_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

GALLERY_PROBE_AVAILABLE = os.path.exists(SPLIT_GALLERY) and os.path.exists(SPLIT_PROBE)
if not GALLERY_PROBE_AVAILABLE:
    print("KHONG tim thay du lieu gallery/probe tai:")
    print(f"  Gallery: {SPLIT_GALLERY}")
    print(f"  Probe:   {SPLIT_PROBE}")
    print("Day la du lieu da tao o Notebook 01 (Buoc 3, Cell 10) - chay Notebook 01 truoc, "
          "hoac kiem tra lai BASE_DIR neu chay tren may khac.")
else:
    n_gallery_people = len(os.listdir(SPLIT_GALLERY))
    n_probe_people = len(os.listdir(SPLIT_PROBE))
    print(f"Da tim thay du lieu: {n_gallery_people} nguoi trong gallery, {n_probe_people} nguoi trong probe")

Da tim thay du lieu: 65 nguoi trong gallery, 65 nguoi trong probe


In [4]:
# ===== CELL A3: Kiem tra GPU (PyTorch cho FaceNet, ONNX Runtime cho insightface, OpenCV cho SFace) =====
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
device = "cuda" if torch.cuda.is_available() else "cpu"

import onnxruntime as ort
print(f"\nONNX Runtime providers: {ort.get_available_providers()}")
if "CUDAExecutionProvider" not in ort.get_available_providers():
    print("CANH BAO: khong co CUDAExecutionProvider - ArcFace / MobileFaceNet / antelopev2 se chay CPU.")
else:
    print("CUDAExecutionProvider co san - cac model insightface se chay GPU.")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070

ONNX Runtime providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'CPUExecutionProvider']
CUDAExecutionProvider co san - cac model insightface se chay GPU.


In [5]:
# ===== CELL A4: Tai model 1 - ArcFace (qua insightface, buffalo_l) =====
import cv2
import numpy as np
from insightface.app import FaceAnalysis

providers = ["CUDAExecutionProvider", "CPUExecutionProvider"]
arcface_app = FaceAnalysis(name="buffalo_l", providers=providers)
arcface_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.3)  # det_thresh giong het config.py

PAD_RATIO = 0.4  # giong het face_recognition_service.py


def get_embedding_arcface(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return None
    h, w = img_bgr.shape[:2]
    pad_h, pad_w = int(h * PAD_RATIO), int(w * PAD_RATIO)
    padded = cv2.copyMakeBorder(img_bgr, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REFLECT)
    faces = arcface_app.get(padded)
    if not faces:
        return None
    best = max(faces, key=lambda f: f.det_score)
    emb = best.embedding.astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

print(f"Da tai ArcFace (buffalo_l), providers: {arcface_app.det_model.session.get_providers()}")

d:\face-security-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Admin/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'

In [6]:
# ===== CELL A5: Tai model 2 - FaceNet (facenet-pytorch, InceptionResnetV1, pretrained vggface2) =====
from facenet_pytorch import InceptionResnetV1

facenet_model = InceptionResnetV1(pretrained="vggface2").eval().to(device)


def get_embedding_facenet(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return None
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_resized = cv2.resize(img_rgb, (160, 160))
    img_tensor = torch.from_numpy(img_resized).float().permute(2, 0, 1).unsqueeze(0)
    img_tensor = (img_tensor - 127.5) / 128.0
    img_tensor = img_tensor.to(device)

    with torch.no_grad():
        emb = facenet_model(img_tensor).cpu().numpy()[0].astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

print(f"Da tai FaceNet (InceptionResnetV1, pretrained=vggface2), dang chay tren: {device}")

Da tai FaceNet (InceptionResnetV1, pretrained=vggface2), dang chay tren: cuda


In [7]:
# ===== CELL A6: Tai model 3 - MobileFaceNet (qua insightface, buffalo_sc) =====
mobilefacenet_app = FaceAnalysis(name="buffalo_sc", providers=providers)
mobilefacenet_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.3)


def get_embedding_mobilefacenet(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return None
    h, w = img_bgr.shape[:2]
    pad_h, pad_w = int(h * PAD_RATIO), int(w * PAD_RATIO)
    padded = cv2.copyMakeBorder(img_bgr, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REFLECT)
    faces = mobilefacenet_app.get(padded)
    if not faces:
        return None
    best = max(faces, key=lambda f: f.det_score)
    emb = best.embedding.astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

print(f"Da tai MobileFaceNet (buffalo_sc), providers: {mobilefacenet_app.det_model.session.get_providers()}")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\Admin/.insightface\models\buffalo_sc\det_500m.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], 

In [8]:
# ===== FIX antelopev2: tải model vào D:\face-security-system\ai\models\antelopev2 =====
import os
import shutil
import zipfile
import requests
from pathlib import Path

BASE_DIR = r"D:\face-security-system"
AI_DIR = Path(BASE_DIR) / "ai"
MODELS_DIR = AI_DIR / "models"

TARGET_DIR = MODELS_DIR / "antelopev2"
ZIP_PATH = MODELS_DIR / "antelopev2.zip"
TMP_DIR = MODELS_DIR / "_antelopev2_tmp"

# Nguồn zip công khai
ZIP_URLS = [
    "https://huggingface.co/vladmandic/insightface-faceanalysis/resolve/main/antelopev2.zip",
]

# Fallback: tải từng file nếu zip lỗi
FILE_BASE = "https://huggingface.co/Aitrepreneur/insightface/resolve/main/models/antelopev2"
ALL_FILES = [
    "1k3d68.onnx",
    "2d106det.onnx",
    "genderage.onnx",
    "glintr100.onnx",
    "scrfd_10g_bnkps.onnx",
]

REQUIRED_COMMON = [
    "1k3d68.onnx",
    "2d106det.onnx",
    "genderage.onnx",
    "glintr100.onnx",
]

DETECTION_FILES = [
    "det_10g.onnx",
    "scrfd_10g_bnkps.onnx",
]

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}


def remove_path(p: Path):
    if p.is_dir():
        shutil.rmtree(p, ignore_errors=True)
    elif p.exists():
        p.unlink()


def download_to_file(url: str, dest: Path):
    print(f"Đang tải: {url}")
    with requests.get(url, stream=True, timeout=180, headers=HEADERS) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        downloaded = 0

        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total:
                        pct = downloaded * 100 / total
                        print(
                            f"\r{pct:.1f}%  {downloaded / 1024 / 1024:.1f}/{total / 1024 / 1024:.1f} MB",
                            end=""
                        )

        if total:
            print()

    size = dest.stat().st_size
    print(f"Đã lưu: {dest} ({size / 1024 / 1024:.1f} MB)")

    if size == 0:
        raise RuntimeError(f"File tải về bị rỗng: {dest}")


def has_detection(model_dir: Path) -> bool:
    return any((model_dir / f).exists() for f in DETECTION_FILES)


def is_valid_model_dir(model_dir: Path) -> bool:
    return has_detection(model_dir) and (model_dir / "glintr100.onnx").exists()


def verify_target_dir(model_dir: Path) -> bool:
    missing = []

    for f in REQUIRED_COMMON:
        if not (model_dir / f).exists():
            missing.append(f)

    if not has_detection(model_dir):
        missing.append("det_10g.onnx hoặc scrfd_10g_bnkps.onnx")

    if missing:
        print("Thiếu file:", missing)
        return False

    print("OK: thư mục antelopev2 hợp lệ.")
    for f in sorted(model_dir.glob("*.onnx")):
        print(f"{f.name:<25} {f.stat().st_size / 1024 / 1024:.1f} MB")

    return True


def download_zip_and_extract():
    last_err = None

    for url in ZIP_URLS:
        try:
            remove_path(ZIP_PATH)
            remove_path(TMP_DIR)

            download_to_file(url, ZIP_PATH)

            if not zipfile.is_zipfile(ZIP_PATH):
                raise RuntimeError("File tải về không phải ZIP hợp lệ.")

            TMP_DIR.mkdir(parents=True, exist_ok=True)

            print("Đang giải nén...")
            with zipfile.ZipFile(ZIP_PATH, "r") as z:
                z.extractall(TMP_DIR)

            # Tìm thư mục thật sự chứa model
            candidates = []
            for det_name in DETECTION_FILES:
                candidates.extend(TMP_DIR.rglob(det_name))

            if not candidates:
                raise RuntimeError("Không tìm thấy detection model trong zip.")

            model_dir = candidates[0].parent

            # Nếu cần, đi ngược lên đến thư mục chứa đầy đủ model
            while model_dir != TMP_DIR and not is_valid_model_dir(model_dir):
                model_dir = model_dir.parent

            if not is_valid_model_dir(model_dir):
                model_dir = candidates[0].parent

            print("Thư mục model trong zip:", model_dir)

            remove_path(TARGET_DIR)

            if model_dir == TMP_DIR:
                shutil.move(str(TMP_DIR), str(TARGET_DIR))
            else:
                shutil.move(str(model_dir), str(TARGET_DIR))
                shutil.rmtree(TMP_DIR, ignore_errors=True)

            return

        except Exception as e:
            last_err = e
            print(f"Lỗi với {url}: {e}")
            remove_path(TMP_DIR)
            remove_path(ZIP_PATH)

    raise RuntimeError(f"Không tải/giải nén được antelopev2.zip. Lỗi cuối: {last_err}")


def download_individual_files():
    print("Bắt đầu fallback: tải từng file ONNX.")
    remove_path(TARGET_DIR)
    TARGET_DIR.mkdir(parents=True, exist_ok=True)

    for fname in ALL_FILES:
        url = f"{FILE_BASE}/{fname}"
        dest = TARGET_DIR / fname
        download_to_file(url, dest)


# ===== Chạy logic chính =====
print("Thư mục model mục tiêu:", TARGET_DIR)

MODELS_DIR.mkdir(parents=True, exist_ok=True)

remove_path(TARGET_DIR)
remove_path(TMP_DIR)
remove_path(ZIP_PATH)

ok = False

try:
    download_zip_and_extract()
    ok = verify_target_dir(TARGET_DIR)
except Exception as e:
    print("Tải zip thất bại:", e)

if not ok:
    print("Chuyển sang tải từng file từ HuggingFace...")
    try:
        download_individual_files()
        ok = verify_target_dir(TARGET_DIR)
    except Exception as e:
        raise RuntimeError(f"Tải antelopev2 thất bại: {e}")

if not ok:
    raise RuntimeError("antelopev2 vẫn chưa hợp lệ. Kiểm tra lại thư mục và mạng.")

Thư mục model mục tiêu: D:\face-security-system\ai\models\antelopev2
Đang tải: https://huggingface.co/vladmandic/insightface-faceanalysis/resolve/main/antelopev2.zip
100.0%  344.0/344.0 MB
Đã lưu: D:\face-security-system\ai\models\antelopev2.zip (344.0 MB)
Đang giải nén...
Thư mục model trong zip: D:\face-security-system\ai\models\_antelopev2_tmp\antelopev2
OK: thư mục antelopev2 hợp lệ.
1k3d68.onnx               137.0 MB
2d106det.onnx             4.8 MB
genderage.onnx            1.3 MB
glintr100.onnx            248.6 MB
scrfd_10g_bnkps.onnx      16.1 MB


In [9]:
import os
from pathlib import Path

insightface_root = Path(os.path.expanduser("~/.insightface"))
model_dir = insightface_root / "models" / "antelopev2"

print("InsightFace root:", insightface_root)
print("antelopev2 dir:", model_dir)

if not model_dir.exists():
    print("=> Chưa có thư mục antelopev2.")
else:
    files = sorted(model_dir.glob("*"))
    if not files:
        print("=> Thư mục antelopev2 tồn tại nhưng RỖNG.")
    else:
        print("=> Danh sách file:")
        for f in files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"{f.name:<25} {size_mb:.2f} MB")

required_files = [
    "det_10g.onnx",
    "glintr100.onnx",
    "genderage.onnx",
    "1k3d68.onnx",
    "2d106det.onnx",
]

missing = [f for f in required_files if not (model_dir / f).exists()]
if missing:
    print("\n=> ĐANG THIẾU CÁC FILE:", missing)
else:
    print("\n=> Có vẻ đủ file antelopev2.")

InsightFace root: C:\Users\Admin\.insightface
antelopev2 dir: C:\Users\Admin\.insightface\models\antelopev2
=> Chưa có thư mục antelopev2.

=> ĐANG THIẾU CÁC FILE: ['det_10g.onnx', 'glintr100.onnx', 'genderage.onnx', '1k3d68.onnx', '2d106det.onnx']


In [10]:
# ===== CELL A7: Tai model 4 - antelopev2 (qua insightface, ResNet100@Glint360K) =====
# antelopev2 la model pack manh hon, cung API FaceAnalysis nhu buffalo_l / buffalo_sc.

# AI_DIR da duoc dinh nghia o Cell A2:
# AI_DIR = D:\face-security-system\ai
# InsightFace se tim model tai:
# D:\face-security-system\ai\models\antelopev2

antelopev2_app = FaceAnalysis(
    name="antelopev2",
    root=AI_DIR,
    providers=providers
)

antelopev2_app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.3)


def get_embedding_antelopev2(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return None
    h, w = img_bgr.shape[:2]
    pad_h, pad_w = int(h * PAD_RATIO), int(w * PAD_RATIO)
    padded = cv2.copyMakeBorder(img_bgr, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REFLECT)
    faces = antelopev2_app.get(padded)
    if not faces:
        return None
    best = max(faces, key=lambda f: f.det_score)
    emb = best.embedding.astype(np.float32)
    return emb / (np.linalg.norm(emb) + 1e-10)

print(f"Da tai antelopev2, providers: {antelopev2_app.det_model.session.get_providers()}")

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: D:\face-security-system\ai\models\antelopev2\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'

In [11]:
# ===== CELL A8: Tai model 5 - AdaFace (UniFace, IR_101) =====
# UniFace public docs cho phep dung AdaFace voi providers=['CPUExecutionProvider'] hoac
# providers GPU neu moi truong ho tro.
from uniface.recognition import AdaFace
from uniface.constants import AdaFaceWeights

adaface_model = AdaFace(model_name=AdaFaceWeights.IR_101, providers=providers)


def _face_landmarks_from_insightface(img_bgr):
    # Dung ArcFace detector lam detector dung chung de lay landmarks cho AdaFace.
    if img_bgr is None or img_bgr.size == 0:
        return None
    h, w = img_bgr.shape[:2]
    pad_h, pad_w = int(h * PAD_RATIO), int(w * PAD_RATIO)
    padded = cv2.copyMakeBorder(img_bgr, pad_h, pad_h, pad_w, pad_w, cv2.BORDER_REFLECT)
    faces = arcface_app.get(padded)
    if not faces:
        return None, None
    best = max(faces, key=lambda f: f.det_score)
    landmarks = getattr(best, "landmark", None)
    if landmarks is None:
        landmarks = getattr(best, "landmarks", None)
    return padded, landmarks


def get_embedding_adaface(img_bgr):
    padded, landmarks = _face_landmarks_from_insightface(img_bgr)
    if padded is None or landmarks is None:
        return None
    emb = adaface_model.get_normalized_embedding(padded, landmarks)
    emb = np.asarray(emb, dtype=np.float32).reshape(-1)
    return emb / (np.linalg.norm(emb) + 1e-10)

print("Da tai AdaFace (UniFace, IR_101)")

Da tai AdaFace (UniFace, IR_101)


In [12]:
# ===== CELL A9: Tai model 6 - SFace (OpenCV FaceRecognizerSF) =====
import urllib.request

SFACE_MODEL_URL = "https://huggingface.co/opencv/opencv_zoo/resolve/main/models/face_recognition_sface/face_recognition_sface_2021dec.onnx"
SFACE_MODEL_DIR = os.path.join(MODELS_DIR, "opencv", "sface")
os.makedirs(SFACE_MODEL_DIR, exist_ok=True)
SFACE_MODEL_PATH = os.path.join(SFACE_MODEL_DIR, "face_recognition_sface_2021dec.onnx")

if not os.path.exists(SFACE_MODEL_PATH):
    print("Dang tai SFace ONNX model...")
    urllib.request.urlretrieve(SFACE_MODEL_URL, SFACE_MODEL_PATH)

sface_model = cv2.FaceRecognizerSF.create(SFACE_MODEL_PATH, "")


def get_embedding_sface(img_bgr):
    if img_bgr is None or img_bgr.size == 0:
        return None
    # SFace expects aligned face input. Gallery/probe cua notebook nay da la crop/aligned 112x112.
    img_resized = cv2.resize(img_bgr, (112, 112))
    feat = sface_model.feature(img_resized).reshape(-1).astype(np.float32)
    return feat / (np.linalg.norm(feat) + 1e-10)

print(f"Da tai SFace model tu: {SFACE_MODEL_PATH}")

Da tai SFace model tu: D:\face-security-system\ai\models\opencv\sface\face_recognition_sface_2021dec.onnx


In [13]:
# ===== CELL A10: Gom 6 ham lai, dat ten thong nhat de vong lap Phan B/C dung chung =====
EMBEDDING_FUNCS = {
    "ArcFace (buffalo_l)": get_embedding_arcface,
    "FaceNet (vggface2)": get_embedding_facenet,
    "MobileFaceNet (buffalo_sc)": get_embedding_mobilefacenet,
    "antelopev2": get_embedding_antelopev2,
    "AdaFace (IR_101)": get_embedding_adaface,
    "SFace": get_embedding_sface,
}

print(f"Da san sang {len(EMBEDDING_FUNCS)} model: {list(EMBEDDING_FUNCS.keys())}")

Da san sang 6 model: ['ArcFace (buffalo_l)', 'FaceNet (vggface2)', 'MobileFaceNet (buffalo_sc)', 'antelopev2', 'AdaFace (IR_101)', 'SFace']


## Phan B - Xay gallery embeddings cho ca 6 model

Giong het logic Notebook 02 (Buoc 2): moi nguoi co the co nhieu anh gallery -> tinh
embedding tung anh roi lay **trung binh** (mean) + chuan hoa lai, thay vi chi dung 1 anh
- lam gallery on dinh hon voi anh ngoai le (goc mat, anh sang khac thuong).

Chay rieng cho **tung model** vi khong gian embedding cua cac model khong lien quan gi
den nhau (khong the "dung chung 1 gallery" cho nhieu encoder).

In [15]:
# ===== VERIFY FACENET DEVICE =====
import torch
import numpy as np

print("device variable:", device)
print("FaceNet model device:", next(facenet_model.parameters()).device)

dummy = torch.randint(
    0, 255,
    size=(1, 3, 160, 160),
    dtype=torch.float32,
    device=torch.device(device)
)

dummy = (dummy - 127.5) / 128.0

with torch.inference_mode():
    out = facenet_model(dummy)

print("FaceNet output device:", out.device)

if torch.cuda.is_available():
    print("CUDA memory allocated:", torch.cuda.memory_allocated() / 1024 / 1024, "MB")

assert out.device.type == "cuda", "FaceNet output KHONG nam tren CUDA"
print("OK: FaceNet dang chay GPU.")

device variable: cuda
FaceNet model device: cuda:0
FaceNet output device: cuda:0
CUDA memory allocated: 115.99169921875 MB
OK: FaceNet dang chay GPU.


In [16]:
# ===== CELL B1: Xay gallery embeddings cho ca 6 model =====
galleries = {model_name: {} for model_name in EMBEDDING_FUNCS}
gallery_build_log = []

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang (xem Cell A2)")
else:
    for model_name, embed_fn in EMBEDDING_FUNCS.items():
        print(f"\n--- Dang xay gallery cho {model_name} ---")
        n_people_done = 0

        for person in sorted(os.listdir(SPLIT_GALLERY)):
            person_dir = os.path.join(SPLIT_GALLERY, person)
            embeddings = []
            for fname in os.listdir(person_dir):
                img = cv2.imread(os.path.join(person_dir, fname))
                emb = embed_fn(img)
                if emb is not None:
                    embeddings.append(emb)

            if embeddings:
                mean_emb = np.mean(embeddings, axis=0)
                mean_emb = mean_emb / (np.linalg.norm(mean_emb) + 1e-10)
                galleries[model_name][person] = mean_emb
                n_people_done += 1

            gallery_build_log.append({
                "model": model_name, "person": person,
                "n_images": len(os.listdir(person_dir)), "n_embeddings_ok": len(embeddings),
            })

        print(f"  Da xay gallery: {n_people_done}/{len(os.listdir(SPLIT_GALLERY))} nguoi co it nhat 1 anh embedding thanh cong")

    gallery_log_df = pd.DataFrame(gallery_build_log)
    gallery_log_df.to_csv(os.path.join(RESULTS_DIR, "recognition_comparison_gallery_build_log.csv"), index=False)


--- Dang xay gallery cho ArcFace (buffalo_l) ---
  Da xay gallery: 65/65 nguoi co it nhat 1 anh embedding thanh cong

--- Dang xay gallery cho FaceNet (vggface2) ---
  Da xay gallery: 65/65 nguoi co it nhat 1 anh embedding thanh cong

--- Dang xay gallery cho MobileFaceNet (buffalo_sc) ---
  Da xay gallery: 65/65 nguoi co it nhat 1 anh embedding thanh cong

--- Dang xay gallery cho antelopev2 ---
  Da xay gallery: 65/65 nguoi co it nhat 1 anh embedding thanh cong

--- Dang xay gallery cho AdaFace (IR_101) ---
  Da xay gallery: 0/65 nguoi co it nhat 1 anh embedding thanh cong

--- Dang xay gallery cho SFace ---
  Da xay gallery: 65/65 nguoi co it nhat 1 anh embedding thanh cong


In [ ]:
# ===== CELL B2: Kiem tra nhanh so nguoi bi loai o tung model (anh crop san nhung van khong tim ra mat) =====

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    coverage = gallery_log_df.groupby("model").apply(
        lambda g: (g["n_embeddings_ok"] > 0).sum()
    ).rename("n_people_with_gallery")
    print("So nguoi co gallery embedding thanh cong (tren tong so nguoi trong SPLIT_GALLERY):\n")
    print(coverage.to_string())
    print(f"\nTong so nguoi trong SPLIT_GALLERY: {len(os.listdir(SPLIT_GALLERY))}")
    print("\nNeu MobileFaceNet (model nhe, detector SCRFD-500MF nho hon buffalo_l) co so "
          "nguoi thap hon ArcFace - day la diem dang luu y cho bao cao: danh doi giua toc "
          "do/kich thuoc model va do nhay phat hien mat trong dieu kien anh khong ly tuong.")

## Phan C - FAR / FRR / EER cho ca 6 model (dung y het phuong phap Notebook 04 - Phan A)

**Genuine (Probe):** anh cua nguoi DA co trong gallery -> do FRR (bi tu choi nham).
**Impostor:** lay THEM 1 batch LFW khac (nhung nguoi CHUA tung duoc dua vao gallery) -> do FAR
(bi nhan nham la nguoi quen). Dung `MAX_STRANGER_IMAGES=500` giong Notebook 04 de thoi gian
chay hop ly - tang len neu can so lieu on dinh hon cho bao cao chinh thuc.

**Luu y ve thang do similarity giua 6 model:** ca 3 ham `get_embedding_*` o tren deu da
L2-normalize embedding, nen `np.dot(a, b)` chinh la **cosine similarity** cho CA 6 model.
Vi vay threshold sweep 0->1 dung chung duoc cho ca 3, khong can quy doi rieng.

In [ ]:
# ===== CELL C1: Ham so khop gallery (dung chung cho ca 6 model, giong match_against_gallery Notebook 02/04) =====

def match_against_gallery(query_emb, gallery_dict):
    if query_emb is None or not gallery_dict:
        return None, -1.0
    best_person, best_score = None, -1.0
    for person, gallery_emb in gallery_dict.items():
        score = float(np.dot(query_emb, gallery_emb))
        if score > best_score:
            best_person, best_score = person, score
    return best_person, best_score


print("Da dinh nghia match_against_gallery()")

In [ ]:
# ===== CELL C2: Tinh diem GENUINE (Probe set) cho ca 6 model =====
genuine_dfs = {}

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    for model_name, embed_fn in EMBEDDING_FUNCS.items():
        print(f"Dang tinh genuine scores cho {model_name}...")
        rows = []
        for person in sorted(os.listdir(SPLIT_PROBE)):
            person_dir = os.path.join(SPLIT_PROBE, person)
            for fname in os.listdir(person_dir):
                img = cv2.imread(os.path.join(person_dir, fname))
                emb = embed_fn(img)
                best_person, best_score = match_against_gallery(emb, galleries[model_name])
                rows.append([person, fname, best_person, best_score])

        genuine_dfs[model_name] = pd.DataFrame(
            rows, columns=["true_person", "filename", "best_match", "best_score"]
        )
        print(f"  {len(genuine_dfs[model_name])} anh probe da xu ly")

In [ ]:
# ===== CELL C3: Tinh diem IMPOSTOR (LFW - nguoi la that su) cho ca 6 model =====
from sklearn.datasets import fetch_lfw_people

MAX_STRANGER_IMAGES = 500  # giong het Notebook 04, tang neu muon test ky hon (chay lau hon)

impostor_dfs = {}

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    print("Dang tai/doc cache LFW voi min_faces_per_person=1 (moi nguoi, ke ca it anh)...")
    lfw_all = fetch_lfw_people(min_faces_per_person=1, resize=1.0, color=True, download_if_missing=True)

    for model_name, embed_fn in EMBEDDING_FUNCS.items():
        enrolled_people = set(galleries[model_name].keys())
        stranger_name_set = {
            n for n in lfw_all.target_names if n.replace(" ", "_") not in enrolled_people
        }

        print(f"\nDang tinh impostor scores cho {model_name} "
              f"({len(stranger_name_set)} nguoi la kha dung)...")
        rows = []
        count = 0
        for idx in range(lfw_all.images.shape[0]):
            if count >= MAX_STRANGER_IMAGES:
                break
            person_name = lfw_all.target_names[lfw_all.target[idx]]
            if person_name not in stranger_name_set:
                continue

            img_rgb = (lfw_all.images[idx] * 255).astype(np.uint8)
            img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
            emb = embed_fn(img_bgr)
            best_person, best_score = match_against_gallery(emb, galleries[model_name])
            rows.append([person_name, best_person, best_score])
            count += 1

        impostor_dfs[model_name] = pd.DataFrame(rows, columns=["true_person", "best_match", "best_score"])
        print(f"  {len(impostor_dfs[model_name])} anh impostor da xu ly")

In [ ]:
# ===== CELL C4: Sweep threshold -> FAR/FRR/EER cho tung model =====
thresholds = np.arange(0.0, 1.01, 0.02)
roc_dfs = {}
eer_summary_rows = []

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    for model_name in EMBEDDING_FUNCS.keys():
        genuine_valid = genuine_dfs[model_name].dropna(subset=["best_score"])
        impostor_valid = impostor_dfs[model_name].dropna(subset=["best_score"])

        far_list, frr_list = [], []
        for th in thresholds:
            frr = (
                (genuine_valid["best_score"] < th) |
                ((genuine_valid["best_score"] >= th) & (genuine_valid["best_match"] != genuine_valid["true_person"]))
            ).mean()
            far = (impostor_valid["best_score"] >= th).mean()
            far_list.append(far)
            frr_list.append(frr)

        roc_df = pd.DataFrame({"threshold": thresholds, "FAR": far_list, "FRR": frr_list})
        roc_df["diff"] = (roc_df["FAR"] - roc_df["FRR"]).abs()
        eer_row = roc_df.loc[roc_df["diff"].idxmin()]
        roc_dfs[model_name] = roc_df

        eer_summary_rows.append({
            "model": model_name,
            "EER_threshold": round(float(eer_row["threshold"]), 3),
            "EER (%)": round(float((eer_row["FAR"] + eer_row["FRR"]) / 2 * 100), 2),
            "FAR_at_EER (%)": round(float(eer_row["FAR"] * 100), 2),
            "FRR_at_EER (%)": round(float(eer_row["FRR"] * 100), 2),
            "n_genuine": len(genuine_valid), "n_impostor": len(impostor_valid),
            "n_genuine_no_face": int(genuine_dfs[model_name]["best_score"].isna().sum()),
        })

    eer_summary_df = pd.DataFrame(eer_summary_rows)
    print("Tong ket EER (Equal Error Rate) cho ca 6 model:\n")
    print(eer_summary_df.to_string(index=False))
    eer_summary_df.to_csv(os.path.join(RESULTS_DIR, "recognition_comparison_eer_summary.csv"), index=False)

In [ ]:
# ===== CELL C5: FAR/FRR tai threshold co dinh 0.35 (threshold hien tai cua he thong) cho ca 6 model =====
CURRENT_THRESHOLD = 0.35  # giong het FACE_MATCH_THRESHOLD trong backend/app/config.py

fixed_threshold_rows = []

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    for model_name, roc_df in roc_dfs.items():
        row = roc_df.iloc[(roc_df["threshold"] - CURRENT_THRESHOLD).abs().argmin()]
        fixed_threshold_rows.append({
            "model": model_name, "threshold": CURRENT_THRESHOLD,
            "FAR (%)": round(float(row["FAR"] * 100), 2),
            "FRR (%)": round(float(row["FRR"] * 100), 2),
        })

    fixed_df = pd.DataFrame(fixed_threshold_rows)
    print(f"FAR/FRR tai threshold={CURRENT_THRESHOLD} (gia tri he thong dang dung THAT cho "
          f"ArcFace buffalo_l - CHI mang tinh tham khao cho FaceNet/MobileFaceNet vi 2 model "
          f"nay chua duoc tune threshold rieng):\n")
    print(fixed_df.to_string(index=False))
    print("\nLuu y quan trong: threshold 0.35 duoc tune RIENG cho ArcFace buffalo_l (xem "
          "Notebook 02/04) - so sanh FAR/FRR cua FaceNet/MobileFaceNet TAI CUNG threshold "
          "nay la KHONG cong bang cho 2 model do. So sanh cong bang hon la nhin vao EER "
          "(Cell C4) - moi model duoc danh gia tai threshold TOI UU cua rieng no.")

In [ ]:
# ===== CELL C6: Bieu do FAR/FRR theo threshold - 6 subplot (2x3) =====
import matplotlib.pyplot as plt

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    n_models = len(list(EMBEDDING_FUNCS.keys()))
    ncols = 3
    nrows = (n_models + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 5 * nrows), sharey=True)
    axes = np.array(axes).reshape(-1)

    for ax, model_name in zip(axes, EMBEDDING_FUNCS.keys()):
        roc_df = roc_dfs[model_name]
        eer_row = eer_summary_df[eer_summary_df["model"] == model_name].iloc[0]

        ax.plot(roc_df["threshold"], roc_df["FAR"] * 100, label="FAR (%)", color="red")
        ax.plot(roc_df["threshold"], roc_df["FRR"] * 100, label="FRR (%)", color="blue")
        ax.axvline(eer_row["EER_threshold"], color="green", linestyle=":",
                    label=f"EER threshold ({eer_row['EER_threshold']:.2f})")
        ax.set_xlabel("Threshold")
        ax.set_title(f"{model_name}\nEER = {eer_row['EER (%)']:.2f}%")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    for ax in axes[n_models:]:
        ax.axis("off")

    axes[0].set_ylabel("Ty le loi (%)")
    plt.tight_layout()
    plot_path = os.path.join(RESULTS_DIR, "recognition_comparison_far_frr.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Da luu bieu do: {plot_path}")

## Phan D - Do toc do trich xuat embedding

Do thoi gian trich 1 embedding tren anh da crop san (khong tinh thoi gian detect nguoi/mat
tu frame goc - do rieng o Notebook 05). **Ca 6 model deu chay GPU trong notebook nay**
(khac ban dau du dinh dung Dlib CPU-only) - xem canh bao chi tiet o dau notebook - nen so
sanh FPS lan nay phan anh dung su khac biet ve KIEN TRUC (ResNet lon vs Inception vs
MobileNet nhe), khong bi nhieu boi viec 1 model bat buoc chay CPU.

In [ ]:
# ===== CELL D1: Do thoi gian trich embedding tren N anh probe, cho ca 6 model =====
import time

N_WARMUP = 5
N_TIMED = 100

speed_rows = []

if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - gallery/probe chua san sang")
else:
    timing_images = []
    for person in sorted(os.listdir(SPLIT_PROBE)):
        for fname in os.listdir(os.path.join(SPLIT_PROBE, person)):
            img = cv2.imread(os.path.join(SPLIT_PROBE, person, fname))
            if img is not None:
                timing_images.append(img)
            if len(timing_images) >= N_WARMUP + N_TIMED:
                break
        if len(timing_images) >= N_WARMUP + N_TIMED:
            break

    for model_name, embed_fn in EMBEDDING_FUNCS.items():
        print(f"Dang do toc do {model_name}...")

        for img in timing_images[:N_WARMUP]:
            embed_fn(img)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start = time.time()
        for img in timing_images[N_WARMUP:N_WARMUP + N_TIMED]:
            embed_fn(img)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.time() - start

        n_processed = len(timing_images[N_WARMUP:N_WARMUP + N_TIMED])
        fps = n_processed / elapsed if elapsed > 0 else 0.0
        ms_per_image = (elapsed / n_processed) * 1000 if n_processed > 0 else 0.0

        speed_rows.append({
            "model": model_name, "fps": round(fps, 2), "ms_per_image": round(ms_per_image, 2),
        })

    speed_df = pd.DataFrame(speed_rows)
    print(f"\nToc do trich embedding (batch size = 1, tren anh da crop san, "
          f"{'GPU: ' + torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}):\n")
    print(speed_df.to_string(index=False))
    speed_df.to_csv(os.path.join(RESULTS_DIR, "recognition_comparison_speed.csv"), index=False)

In [ ]:
# ===== CELL D2: Bieu do toc do =====
if not GALLERY_PROBE_AVAILABLE:
    print("Bo qua Cell nay - chua co ket qua toc do")
else:
    plt.figure(figsize=(8, 5))
    colors = plt.cm.tab10(np.linspace(0, 1, len(speed_df)))
    bars = plt.bar(speed_df["model"], speed_df["fps"], color=colors)
    plt.ylabel("FPS (embedding/giay, batch size = 1)")
    plt.title("Toc do trich embedding - " + (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"))
    plt.xticks(rotation=15, ha="right")
    for bar, fps in zip(bars, speed_df["fps"]):
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{fps:.1f}",
                ha="center", va="bottom")
    plt.grid(alpha=0.3, axis="y")
    plt.tight_layout()
    plot_path = os.path.join(RESULTS_DIR, "recognition_comparison_speed.png")
    plt.savefig(plot_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"Da luu bieu do: {plot_path}")

## Phan E - Tong ket

**Ket qua luu trong `ai/dataset/evaluation_results/`:**
- `recognition_comparison_gallery_build_log.csv` - so anh/embedding thanh cong tung nguoi, tung model
- `recognition_comparison_eer_summary.csv` - EER, threshold toi uu, FAR/FRR tai EER cho ca 6 model
- `recognition_comparison_speed.csv` - FPS/ms-per-embedding
- `recognition_comparison_far_frr.png`, `recognition_comparison_speed.png` - bieu do

**Cach doc ket qua cho bao cao (Chuong 4):**
- So sanh chinh nen dua tren **EER** (Cell C4) - day la tieu chi cong bang vi moi model
  duoc xet tai threshold toi uu CUA RIENG NO, khac voi bang FAR/FRR tai threshold co dinh
  0.35 (Cell C5) chi mang tinh tham khao.
- Neu ArcFace (buffalo_l) co EER thap nhat - day la bang chung dinh luong cho model baseline
  dang duoc dung trong backend.
- Neu antelopev2 / AdaFace co EER thap hon ArcFace nhung FPS thap hon - day la minh chung
  cho trade-off giua do chinh xac va toc do.
- Neu MobileFaceNet hoac SFace co FPS cao hon ro ret nhung EER cao hon - day la minh chung
  cho huong lightweight / edge deployment.
- Neu FaceNet cho ket qua khac biet ro so voi cac model ArcFace-style - do la diem de phan
  tich khac nhau ve loss function (Triplet Loss vs ArcFace / AdaFace).

**Luu y quan trong khi doc bang so lieu:**
- `MAX_STRANGER_IMAGES=500` (Cell C3, giong Notebook 04) chi la mot phan cua LFW - tang len
  neu muon EER on dinh hon cho bao cao chinh thuc, chap nhan doi lau hon.
- Gallery/probe dang dung la **LFW qua Notebook 01** (du lieu demo) - khi co du lieu enrollment
  that cua do an, nen chay lai notebook nay tren du lieu that truoc khi ket luan cuoi cung.
- Cac model co the dung pipeline detect/alignment khac nhau (insightface / UniFace / OpenCV),
  nen khi bao cao can ghi ro rang model nao dang benchmark theo encoder don thuần, model nao
  phai dua vao detector/landmark trung gian de trich embedding.